In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
import statsmodels.graphics.tsaplots
from statsmodels.tsa.stattools import adfuller
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
import statistics

In [3]:
data = pd.read_csv('Data/preprocessed/NP15_da_series.csv')
data['begin_time'] = pd.to_datetime(data['begin_time'])

We first take about 80% of the data from the start as training data.

In [4]:
data_train=data.iloc[:35000]
data_train.head(30)

,begin_time,NP-15 LMP,NP-15 LMP 24hrdiffs,NP-15 LMP Prev_day_price
0,2020-01-01 00:00:00,32.76137,NaN,NaN
1,2020-01-01 01:00:00,30.90384,NaN,NaN
2,2020-01-01 02:00:00,32.15974,NaN,NaN
3,2020-01-01 03:00:00,31.24182,NaN,NaN
4,2020-01-01 04:00:00,30.98365,NaN,NaN
5,2020-01-01 05:00:00,33.17688,NaN,NaN
6,2020-01-01 06:00:00,33.87663,NaN,NaN
7,2020-01-01 07:00:00,32.69939,NaN,NaN
8,2020-01-01 08:00:00,29.30171,NaN,NaN
9,2020-01-01 09:00:00,23.50110,NaN,NaN


We aim to perform cross validation for various models across time series splits. We customise cross validation to get validation for a variety of days (i.e. not all of them should be in December).

In [5]:
# Get data between specified dates
def filter_times(df, time1, time2):
    return df.apply(lambda x: (time1 <= x['begin_time']) and (x['begin_time'] <= time2) , axis=1)

In [78]:
def crossValidate(year, n_splits, test_size, seas_order, non_seas_order):
    ts_split = TimeSeriesSplit(n_splits=n_splits, test_size=test_size)
    mse_list = []
    mse_base_list = []

    print(f"Seasonal order - {seas_order}, Non-seasonal order - {non_seas_order}")

    for j in range(1,7):
        time1 = datetime.datetime(year-1, 1, 1, 0, 0, 0)+datetime.timedelta(days=60*j)
        time2 = datetime.datetime(year, 1, 1, 0, 0, 0)+datetime.timedelta(days=60*j)
        df = data_train[filter_times(data_train, time1, time2)].copy().reset_index(drop=True)
        for i, (train_index, test_index) in enumerate(ts_split.split(df)):
            model = SARIMAX(endog=df['NP-15 LMP'].loc[train_index], order=non_seas_order, seasonal_order=seas_order)
            model_fit = model.fit()
            forecast = model_fit.forecast(test_size).to_frame(name='predicted')
            eval_df = pd.merge(df[['NP-15 LMP', 'NP-15 LMP Prev_day_price']].loc[test_index].copy(), forecast, left_index=True, right_index=True, how='inner')
            #eval_df = eval_df[eval_df.apply(lambda x: (x['NP-15 LMP'].isna()==False) and (x['NP-15 LMP Prev_day_price'].isna()==False))]
            eval_df = eval_df[(eval_df['NP-15 LMP'].isna()==False)]
            eval_df = eval_df[(eval_df['NP-15 LMP Prev_day_price'].isna()==False)]
            if eval_df.empty == False:
                mse = mean_squared_error(eval_df['NP-15 LMP'], eval_df['predicted'])
                mse_base = mean_squared_error(eval_df['NP-15 LMP'], eval_df['NP-15 LMP Prev_day_price'])
                mse_list.append(mse)
                mse_base_list.append(mse_base)
                #print(f"mse for validation in {time2}, fold {i} = {mse}. Baseline = {mse_base}")
            #print(model_fit.summary())
            #mse = mean_squared_error(df['NP-15 LMP'].loc[test_index], model_fit.forecast(test_size))
            #mse_list.append(mse)
            #data_predicted = df.join(pd.concat([df['NP-15 LMP'].loc[train_index].tail(test_size), model_fit.forecast(test_size)]).to_frame(name='predicted'), how='inner')
            #plt.figure(figsize=(18, 4))
            #plt.plot(data_predicted['begin_time'], data_predicted['NP-15 LMP Prev_day_price'] , label='Previous day price')
            #plt.plot(data_predicted['begin_time'], data_predicted['NP-15 LMP'] , label='Actual price')
            #plt.plot(data_predicted['begin_time'], data_predicted['predicted'] , label='Predicted')
            #plt.xticks(rotation=90)
            #plt.legend()
            #plt.show()
        #print(f"Set {j} done.")
    
    print(f"Validation error for {year} = {statistics.fmean(mse_list)}, baseline = {statistics.fmean(mse_base_list)}")

In [70]:
crossValidate(2023, 3, 24, (0,1,0, 24), (0,0,0))

Seasonal order - (0, 1, 0, 24), Non-seasonal order - (0, 0, 0)
mse for validation in 2023-03-02 00:00:00, fold 0 = 796.261671026446. Baseline = 796.2616710264459
mse for validation in 2023-03-02 00:00:00, fold 1 = 1433.0040755400416. Baseline = 1433.0040755400416
mse for validation in 2023-03-02 00:00:00, fold 2 = 115.43257036568336. Baseline = 115.43257036568332
mse for validation in 2023-05-01 00:00:00, fold 0 = 149.25311766787502. Baseline = 149.25311766787502
mse for validation in 2023-05-01 00:00:00, fold 1 = 456.6877630152626. Baseline = 456.6877630152626
mse for validation in 2023-05-01 00:00:00, fold 2 = 412.22370193103325. Baseline = 412.2237019310333
mse for validation in 2023-06-30 00:00:00, fold 0 = 48.03869575279166. Baseline = 48.038695752791675
mse for validation in 2023-06-30 00:00:00, fold 1 = 11.754681925124999. Baseline = 11.754681925124999
mse for validation in 2023-06-30 00:00:00, fold 2 = 102.75106374161665. Baseline = 102.75106374161665
mse for validation in 2023

In [71]:
crossValidate(2023, 3, 24, (0,1,0, 24), (0,1,0))

Seasonal order - (0, 1, 0, 24), Non-seasonal order - (0, 1, 0)
mse for validation in 2023-03-02 00:00:00, fold 0 = 671.5837315051217. Baseline = 796.2616710264459
mse for validation in 2023-03-02 00:00:00, fold 1 = 148.57522759824133. Baseline = 1433.0040755400416
mse for validation in 2023-03-02 00:00:00, fold 2 = 197.099178184483. Baseline = 115.43257036568332
mse for validation in 2023-05-01 00:00:00, fold 0 = 157.17806194940883. Baseline = 149.25311766787502
mse for validation in 2023-05-01 00:00:00, fold 1 = 327.8704688874872. Baseline = 456.6877630152626
mse for validation in 2023-05-01 00:00:00, fold 2 = 103.55069526798337. Baseline = 412.2237019310333
mse for validation in 2023-06-30 00:00:00, fold 0 = 76.63588441749174. Baseline = 48.038695752791675
mse for validation in 2023-06-30 00:00:00, fold 1 = 12.716375216674999. Baseline = 11.754681925124999
mse for validation in 2023-06-30 00:00:00, fold 2 = 104.43823120854995. Baseline = 102.75106374161665
mse for validation in 2023-

In [72]:
crossValidate(2023, 3, 24, (0,1,0, 24), (1,1,0))

Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 1, 0)
mse for validation in 2023-03-02 00:00:00, fold 0 = 492.5502309952658. Baseline = 796.2616710264459
mse for validation in 2023-03-02 00:00:00, fold 1 = 147.97621348262626. Baseline = 1433.0040755400416
mse for validation in 2023-03-02 00:00:00, fold 2 = 476.20561156586683. Baseline = 115.43257036568332
mse for validation in 2023-05-01 00:00:00, fold 0 = 177.0075044323879. Baseline = 149.25311766787502
mse for validation in 2023-05-01 00:00:00, fold 1 = 316.9954418359448. Baseline = 456.6877630152626
mse for validation in 2023-05-01 00:00:00, fold 2 = 105.84541110051357. Baseline = 412.2237019310333
mse for validation in 2023-06-30 00:00:00, fold 0 = 72.98379682293239. Baseline = 48.038695752791675
mse for validation in 2023-06-30 00:00:00, fold 1 = 12.928415838107563. Baseline = 11.754681925124999
mse for validation in 2023-06-30 00:00:00, fold 2 = 97.76303604213751. Baseline = 102.75106374161665
mse for validation in 2023-

In [73]:
crossValidate(2023, 3, 24, (1,1,0, 24), (1,1,0))

Seasonal order - (1, 1, 0, 24), Non-seasonal order - (1, 1, 0)
mse for validation in 2023-03-02 00:00:00, fold 0 = 327.06418819012106. Baseline = 796.2616710264459
mse for validation in 2023-03-02 00:00:00, fold 1 = 195.72626354846713. Baseline = 1433.0040755400416
mse for validation in 2023-03-02 00:00:00, fold 2 = 553.3287597262178. Baseline = 115.43257036568332
mse for validation in 2023-05-01 00:00:00, fold 0 = 108.8139429125517. Baseline = 149.25311766787502
mse for validation in 2023-05-01 00:00:00, fold 1 = 409.4870906235721. Baseline = 456.6877630152626
mse for validation in 2023-05-01 00:00:00, fold 2 = 165.80913500939218. Baseline = 412.2237019310333
mse for validation in 2023-06-30 00:00:00, fold 0 = 29.54961607462863. Baseline = 48.038695752791675
mse for validation in 2023-06-30 00:00:00, fold 1 = 15.883703508021293. Baseline = 11.754681925124999
mse for validation in 2023-06-30 00:00:00, fold 2 = 91.99203159186607. Baseline = 102.75106374161665
mse for validation in 2023-

In [74]:
crossValidate(2023, 3, 48, (1,1,1, 24), (1,1,0))

Seasonal order - (1, 1, 1, 24), Non-seasonal order - (1, 1, 0)
mse for validation in 2023-03-02 00:00:00, fold 0 = 4650.116410389614. Baseline = 1294.9526219834772
mse for validation in 2023-03-02 00:00:00, fold 1 = 438.7809770365828. Baseline = 855.0983670432128
mse for validation in 2023-03-02 00:00:00, fold 2 = 985.2365756210924. Baseline = 774.2183229528624
mse for validation in 2023-05-01 00:00:00, fold 0 = 249.65441137557613. Baseline = 244.72317354354786
mse for validation in 2023-05-01 00:00:00, fold 1 = 534.8444583998563. Baseline = 131.53159426744796
mse for validation in 2023-05-01 00:00:00, fold 2 = 446.1907717091952. Baseline = 434.4557324731479
mse for validation in 2023-06-30 00:00:00, fold 0 = 108.4755032574169. Baseline = 101.40992397491043
mse for validation in 2023-06-30 00:00:00, fold 1 = 182.34313864436308. Baseline = 178.70210716742295
mse for validation in 2023-06-30 00:00:00, fold 2 = 53.714110078102145. Baseline = 57.25287283337082
mse for validation in 2023-10

In [79]:
crossValidate(2023, 3, 24, (0,1,0, 24), (0,0,0))
crossValidate(2023, 3, 24, (0,1,0, 24), (1,0,0))
crossValidate(2023, 3, 24, (0,1,0, 24), (1,0,1))

#crossValidate(2023, 3, 24, (1,1,0, 24), (0,0,0))
crossValidate(2023, 3, 24, (1,1,0, 24), (1,0,0))
crossValidate(2023, 3, 24, (1,1,0, 24), (1,0,1))

#crossValidate(2023, 3, 24, (1,1,1, 24), (0,0,0))
crossValidate(2023, 3, 24, (1,1,1, 24), (1,0,0))
#crossValidate(2023, 3, 24, (1,1,1, 24), (1,0,1))


Seasonal order - (0, 1, 0, 24), Non-seasonal order - (0, 0, 0)
Validation error for 2023 = 245.72216611976225, baseline = 245.72216611976222
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 0, 0)
Validation error for 2023 = 189.13241114895632, baseline = 245.72216611976222
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 0, 1)
Validation error for 2023 = 193.56366703864504, baseline = 245.72216611976222
Seasonal order - (1, 1, 0, 24), Non-seasonal order - (0, 0, 0)
Validation error for 2023 = 233.86901419163982, baseline = 245.72216611976222
Seasonal order - (1, 1, 0, 24), Non-seasonal order - (1, 0, 0)
Validation error for 2023 = 170.26778962701638, baseline = 245.72216611976222
Seasonal order - (1, 1, 0, 24), Non-seasonal order - (1, 0, 1)
Validation error for 2023 = 174.43340681866403, baseline = 245.72216611976222
Seasonal order - (1, 1, 1, 24), Non-seasonal order - (0, 0, 0)
Validation error for 2023 = 241.23334034385732, baseline = 245.72216611976222
Seasonal orde

KeyboardInterrupt: 

In [82]:
crossValidate(2022, 3, 24, (0,1,0, 24), (0,0,0))
crossValidate(2022, 3, 24, (0,1,0, 24), (0,1,0))
crossValidate(2022, 3, 24, (0,1,0, 24), (1,0,0))
crossValidate(2022, 3, 24, (0,1,0, 24), (1,1,0))
crossValidate(2022, 3, 24, (0,1,0, 24), (1,1,1))

#crossValidate(2023, 3, 24, (1,1,0, 24), (0,0,0))
crossValidate(2022, 3, 24, (1,1,0, 24), (0,1,0))
crossValidate(2022, 3, 24, (1,1,0, 24), (1,0,0))
crossValidate(2022, 3, 24, (1,1,0, 24), (1,1,0))

#crossValidate(2023, 3, 24, (1,1,1, 24), (0,0,0))
crossValidate(2022, 3, 24, (1,1,1, 24), (0,1,0))
crossValidate(2022, 3, 24, (1,1,1, 24), (1,0,0))
crossValidate(2022, 3, 24, (1,1,1, 24), (1,1,0))
#crossValidate(2023, 3, 24, (1,1,1, 24), (1,0,1))

Seasonal order - (0, 1, 0, 24), Non-seasonal order - (0, 0, 0)
Validation error for 2022 = 182.92427780702894, baseline = 182.92427780702894
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (0, 1, 0)
Validation error for 2022 = 180.14690194731645, baseline = 182.92427780702894
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 0, 0)
Validation error for 2022 = 168.8991942415595, baseline = 182.92427780702894
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 1, 0)
Validation error for 2022 = 272.2638006640143, baseline = 182.92427780702894
Seasonal order - (0, 1, 0, 24), Non-seasonal order - (1, 1, 1)
Validation error for 2022 = 197.46956054222693, baseline = 182.92427780702894
Seasonal order - (1, 1, 0, 24), Non-seasonal order - (0, 1, 0)
Validation error for 2022 = 173.46236271018216, baseline = 182.92427780702894
Seasonal order - (1, 1, 0, 24), Non-seasonal order - (1, 0, 0)
Validation error for 2022 = 200.78887025763095, baseline = 182.92427780702894
Seasonal order 

KeyboardInterrupt: 

In [83]:
crossValidate(2023, 3, 24, (1,1,1, 24), (0,1,0))

Seasonal order - (1, 1, 1, 24), Non-seasonal order - (0, 1, 0)
Validation error for 2023 = 106.5314669909366, baseline = 245.72216611976222
